# Week 9 Lab - Trees: BST, AVL Trees, Heaps & Priority Queues

**Duration**: 2–3 hours  
**Topics Covered**:  
- Binary Search Tree (BST)  
- AVL Tree (Self-Balancing BST)  
- Heaps (Min-Heap / Max-Heap)  
- Priority Queues  
- Applications in scheduling & real-world data processing  

---

## Learning Objectives

By the end of this lab, you will be able to:

- Implement and manipulate BSTs and AVL Trees
- Construct and use Min/Max Heaps effectively
- Simulate and analyze a real-world **job scheduler** using heaps
- Apply priority queue mechanisms to practical scheduling problems
- Understand trade-offs between tree structures and heap-based systems

## Section 1: BST & AVL Tree Construction and Analysis

### Task 1.1: Custom Binary Search Tree (BST)

Create a `BinarySearchTree` class with the following operations:

- `insert(value)`
- `delete(value)`
- `search(value)` → Returns True/False
- `inorder()` → Returns elements in sorted order
- `is_balanced()` → Checks if tree is height-balanced

In [ ]:
class Node:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None

class BinarySearchTree:
    def __init__(self):
        self.root = None

    def insert(self, value):
        def _insert(node, value):
            if not node:
                return Node(value)
            if value < node.key:
                node.left = _insert(node.left, value)
            elif value > node.key:
                node.right = _insert(node.right, value)
            return node
        self.root = _insert(self.root, value)

    def delete(self, value):
        def _min_value_node(node):
            current = node
            while current.left:
                current = current.left
            return current

        def _delete(node, value):
            if not node:
                return node
            if value < node.key:
                node.left = _delete(node.left, value)
            elif value > node.key:
                node.right = _delete(node.right, value)
            else:
                if not node.left:
                    return node.right
                elif not node.right:
                    return node.left
                temp = _min_value_node(node.right)
                node.key = temp.key
                node.right = _delete(node.right, temp.key)
            return node

        self.root = _delete(self.root, value)

    def search(self, value):
        def _search(node, value):
            if not node:
                return False
            if node.key == value:
                return True
            elif value < node.key:
                return _search(node.left, value)
            else:
                return _search(node.right, value)
        return _search(self.root, value)

    def inorder(self):
        result = []
        def _inorder(node):
            if node:
                _inorder(node.left)
                result.append(node.key)
                _inorder(node.right)
        _inorder(self.root)
        return result

    def is_balanced(self):
        def _check(node):
            if not node:
                return 0, True
            left_height, left_balanced = _check(node.left)
            right_height, right_balanced = _check(node.right)
            balanced = abs(left_height - right_height) <= 1 and left_balanced and right_balanced
            return 1 + max(left_height, right_height), balanced
        _, balanced = _check(self.root)
        return balanced


### Task 1.2: Self-Balancing AVL Tree

Implement an `AVLTree` class with automatic rebalancing.

Your implementation should support:
- Left/Right Rotation
- Balancing after insert/delete
- In-order traversal
- Height tracking

In [ ]:
class AVLNode:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None
        self.height = 1

class AVLTree:
    def __init__(self):
        self.root = None

    def get_height(self, node):
        return node.height if node else 0

    def get_balance(self, node):
        return self.get_height(node.left) - self.get_height(node.right) if node else 0

    def right_rotate(self, y):
        x = y.left
        T2 = x.right
        x.right = y
        y.left = T2
        y.height = 1 + max(self.get_height(y.left), self.get_height(y.right))
        x.height = 1 + max(self.get_height(x.left), self.get_height(x.right))
        return x

    def left_rotate(self, x):
        y = x.right
        T2 = y.left
        y.left = x
        x.right = T2
        x.height = 1 + max(self.get_height(x.left), self.get_height(x.right))
        y.height = 1 + max(self.get_height(y.left), self.get_height(y.right))
        return y

    def insert(self, value):
        def _insert(node, key):
            if not node:
                return AVLNode(key)
            elif key < node.key:
                node.left = _insert(node.left, key)
            elif key > node.key:
                node.right = _insert(node.right, key)
            else:
                return node  # duplicate

            node.height = 1 + max(self.get_height(node.left), self.get_height(node.right))
            balance = self.get_balance(node)

            # LL
            if balance > 1 and key < node.left.key:
                return self.right_rotate(node)
            # RR
            if balance < -1 and key > node.right.key:
                return self.left_rotate(node)
            # LR
            if balance > 1 and key > node.left.key:
                node.left = self.left_rotate(node.left)
                return self.right_rotate(node)
            # RL
            if balance < -1 and key < node.right.key:
                node.right = self.right_rotate(node.right)
                return self.left_rotate(node)

            return node

        self.root = _insert(self.root, value)

    def inorder(self):
        result = []
        def _inorder(node):
            if node:
                _inorder(node.left)
                result.append(node.key)
                _inorder(node.right)
        _inorder(self.root)
        return result

    def get_height_tree(self):
        return self.get_height(self.root)


## Section 2: Heap and Priority Queue

### Task 2.1: Min Heap & Max Heap Implementation

Build your own heap classes from scratch. Do not use Python’s `heapq`.

Each heap class should support:
- `insert(value)`
- `extract()` → Remove root
- `peek()` → Return root
- `heapify(array)` → Build heap in-place
- `visualize()` → Optional: Print tree structure as levels

In [ ]:
class MinHeap:
    def __init__(self):
        self.heap = []

    def insert(self, value):
        self.heap.append(value)
        self._bubble_up(len(self.heap) - 1)

    def _bubble_up(self, index):
        parent = (index - 1) // 2
        while index > 0 and self.heap[parent] > self.heap[index]:
            self.heap[parent], self.heap[index] = self.heap[index], self.heap[parent]
            index = parent
            parent = (index - 1) // 2

    def extract(self):
        if not self.heap:
            return None
        if len(self.heap) == 1:
            return self.heap.pop()
        root = self.heap[0]
        self.heap[0] = self.heap.pop()
        self._heapify(0)
        return root

    def _heapify(self, index):
        smallest = index
        left, right = 2 * index + 1, 2 * index + 2
        if left < len(self.heap) and self.heap[left] < self.heap[smallest]:
            smallest = left
        if right < len(self.heap) and self.heap[right] < self.heap[smallest]:
            smallest = right
        if smallest != index:
            self.heap[index], self.heap[smallest] = self.heap[smallest], self.heap[index]
            self._heapify(smallest)

    def peek(self):
        return self.heap[0] if self.heap else None

    def heapify(self, array):
        self.heap = array[:]
        for i in reversed(range(len(self.heap)//2)):
            self._heapify(i)


class MaxHeap:
    def __init__(self):
        self.heap = []

    def insert(self, value):
        self.heap.append(value)
        self._bubble_up(len(self.heap) - 1)

    def _bubble_up(self, index):
        parent = (index - 1) // 2
        while index > 0 and self.heap[parent] < self.heap[index]:
            self.heap[parent], self.heap[index] = self.heap[index], self.heap[parent]
            index = parent
            parent = (index - 1) // 2

    def extract(self):
        if not self.heap:
            return None
        if len(self.heap) == 1:
            return self.heap.pop()
        root = self.heap[0]
        self.heap[0] = self.heap.pop()
        self._heapify(0)
        return root

    def _heapify(self, index):
        largest = index
        left, right = 2 * index + 1, 2 * index + 2
        if left < len(self.heap) and self.heap[left] > self.heap[largest]:
            largest = left
        if right < len(self.heap) and self.heap[right] > self.heap[largest]:
            largest = right
        if largest != index:
            self.heap[index], self.heap[largest] = self.heap[largest], self.heap[index]
            self._heapify(largest)

    def peek(self):
        return self.heap[0] if self.heap else None

    def heapify(self, array):
        self.heap = array[:]
        for i in reversed(range(len(self.heap)//2)):
            self._heapify(i)


### Task 2.2: Priority Queue Using Heap

Design a custom `PriorityQueue` class with the following:

- Tasks with `task_name`, `priority`, and `timestamp`
- Lower priority number means higher priority
- On same priority, earlier timestamp gets precedence (simulate FIFO)

In [ ]:
import heapq

class Job:
    def __init__(self, task_name, priority, timestamp):
        self.task_name = task_name
        self.priority = priority
        self.timestamp = timestamp

    def __lt__(self, other):
        return (self.priority, self.timestamp) < (other.priority, other.timestamp)

class PriorityQueue:
    def __init__(self):
        self.heap = []

    def enqueue(self, job):
        heapq.heappush(self.heap, job)

    def dequeue(self):
        return heapq.heappop(self.heap) if self.heap else None

    def peek(self):
        return self.heap[0] if self.heap else None

    def is_empty(self):
        return len(self.heap) == 0


## Section 3: Practical Project – Heap-Based Job Scheduler

### Task 3.1: Simulate a Job Scheduler

You are asked to implement a **Job Scheduler** for a server farm. Each job has:

- `job_id`
- `priority` (lower is more urgent)
- `submission_time` (in seconds)
- `duration` (in seconds)

Requirements:

- Maintain a schedule queue using a min-heap priority queue.
- Simulate execution:
  - Pick the job with the highest priority (lowest number)
  - If multiple jobs have the same priority, run the one submitted first
- Keep track of:
  - Total jobs executed
  - Average waiting time
  - Job completion logs

In [ ]:
import heapq

class JobScheduler:
    def __init__(self):
        self.queue = []
        self.logs = []
        self.current_time = 0

    def add_job(self, job_id, priority, submission_time, duration):
        heapq.heappush(self.queue, (priority, submission_time, job_id, duration))

    def run(self):
        while self.queue:
            priority, sub_time, job_id, duration = heapq.heappop(self.queue)
            start_time = max(self.current_time, sub_time)
            end_time = start_time + duration
            wait_time = start_time - sub_time
            self.logs.append({
                "job_id": job_id,
                "start": start_time,
                "end": end_time,
                "wait": wait_time
            })
            self.current_time = end_time

    def print_logs(self):
        for log in self.logs:
            print(f"{log['job_id']}: Start={log['start']}, End={log['end']}, Wait={log['wait']}")

    def average_waiting_time(self):
        if not self.logs:
            return 0
        return sum(log["wait"] for log in self.logs) / len(self.logs)


You may simulate job submissions using random data:

In [ ]:
# Example
# Simulate 20 jobs with random priorities and durations
import random
import time

scheduler = JobScheduler()
for i in range(20):
    scheduler.add_job(
        job_id=f"Job-{i+1}",
        priority=random.randint(1, 5),
        submission_time=i * 2,
        duration=random.randint(1, 10)
    )
scheduler.run()
scheduler.print_logs()
print("Average Waiting Time:", scheduler.average_waiting_time())

## Section 4: Comparative Analysis

### Task 4.1: Compare BST vs AVL vs Heap in Theory

Write a detailed comparison (in markdown cell) covering:

- Insertion/deletion time complexities
- Worst-case tree shapes
- When to prefer which (e.g., AVL vs heap in job scheduling)

## Comparison: BST vs AVL vs Heap

- **BST**
  - Insert/Delete: O(h), where h = height
  - Worst-case: Unbalanced → O(n)
  - Use-case: Ordered traversal, simple lookup
  
- **AVL Tree**
  - Insert/Delete: O(log n)
  - Always balanced
  - Use-case: Lookup-intensive workloads

- **Heap**
  - Insert/Delete: O(log n)
  - No order across elements, only root guaranteed
  - Use-case: Priority queue, scheduling, median maintenance